<a href="https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/exercicios/aula4-exercicio4-de-volta-para-pixels.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 4, exercício 4: De volta para pixels

Pegue a primeira linha do rótulo `000000000009.txt` e converta para pixels no formato `(x1, y1, x2, y2)`, usando o tamanho real da imagem `000000000009.jpg`.

O exercício é da aula [4. Treino com dataset](https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/notebooks/04-treino-com-dataset-proprio.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara os dados, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [1]:
!pip install -q ultralytics

In [2]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

## Preparação

A preparação treina o YOLOv8 por cinco épocas no `coco8`, como na aula 4. Ficam prontos o `df`, com as métricas de cada época, e o `melhor`, que é o modelo treinado.

In [3]:
from pathlib import Path

import pandas as pd
from ultralytics import YOLO
from ultralytics.utils import DATASETS_DIR

model = YOLO("yolov8n.pt")
treino = model.train(
    data="coco8.yaml", epochs=5, imgsz=320, batch=8,
    project="runs", name="coco8", exist_ok=True, plots=False, verbose=False,
)
pasta = Path(treino.save_dir)
df = pd.read_csv(pasta / "results.csv")
df.columns = df.columns.str.strip()
melhor = YOLO(pasta / "weights" / "best.pt")

print(df[["epoch", "train/box_loss", "train/cls_loss", "metrics/mAP50(B)"]].to_string(index=False))

Ultralytics 8.4.180 🚀 Python-3.12.13 torch-2.14.1 CPU (Apple M4)



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                


  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             


...

Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 16.7it/s 0.1s

                   all          4         17      0.638      0.633      0.621      0.461


Speed: 0.2ms preprocess, 12.5ms inference, 0.0ms loss, 0.4ms postprocess per image


 epoch  train/box_loss  train/cls_loss  metrics/mAP50(B)
     1         1.22887         2.08624           0.62097
     2         1.23576         2.45388           0.62097
     3         1.26120         3.28481           0.62097
     4         1.21124         2.74359           0.62097
     5         1.25983         2.01996           0.62097


In [4]:
# @title Correção automática (rode, não precisa ler)
from collections import Counter
from functools import lru_cache

import cv2

_raiz = DATASETS_DIR / "coco8"
_linhas = [
    l.split()
    for f in sorted((_raiz / "labels" / "train").glob("*.txt"))
    for l in f.read_text().splitlines()
    if l.strip()
]
_por_classe = Counter(int(l[0]) for l in _linhas)


def _caixa_em_pixels():
    img = cv2.imread(str(_raiz / "images" / "train" / "000000000009.jpg"))
    h, w = img.shape[:2]
    linha = (_raiz / "labels" / "train" / "000000000009.txt").read_text().splitlines()[0]
    _, x, y, bw, bh = map(float, linha.split())
    return ((x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h)


@lru_cache(None)
def _map50():
    return float(melhor.val(data="coco8.yaml", imgsz=320, verbose=False, plots=False).box.map50)


VERIFICAR = {
    1: lambda r: int(r) == len(list((_raiz / "images" / "train").glob("*.jpg"))),
    2: lambda r: int(r) == len(_linhas),
    3: lambda r: r in {model.names[c] for c, n in _por_classe.items() if n == max(_por_classe.values())},
    4: lambda r: all(abs(a - b) <= 2 for a, b in zip(r, _caixa_em_pixels())),
    5: lambda r: int(r) == int(df.loc[df["train/box_loss"].idxmin(), "epoch"]),
    6: lambda r: abs(float(r) - _map50()) < 1e-3,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Pegue a primeira linha do rótulo `000000000009.txt` e converta para pixels no formato `(x1, y1, x2, y2)`, usando o tamanho real da imagem `000000000009.jpg`.

*Dica:* x1 = (x_centro - largura / 2) * largura_da_imagem, e assim por diante.

Troque o `None` pela sua resposta e rode a célula.

In [5]:
resposta = None

conferir(4, resposta)

Exercício 4: ainda sem resposta.


## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [6]:
raiz = DATASETS_DIR / "coco8"
img = cv2.imread(str(raiz / "images" / "train" / "000000000009.jpg"))
h, w = img.shape[:2]
linha = (raiz / "labels" / "train" / "000000000009.txt").read_text().splitlines()[0]
_, x, y, bw, bh = map(float, linha.split())
resposta = (int((x - bw / 2) * w), int((y - bh / 2) * h), int((x + bw / 2) * w), int((y + bh / 2) * h))

conferir(4, resposta)

Exercício 4: ✅ certo!


## O que fica

Próximo: [Aula 4, exercício 5: A melhor época](https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/exercicios/aula4-exercicio5-a-melhor-epoca.ipynb).